# Meta-learning for few-shot domain adaptation (MAML)

The last stage of the pipeline. We take the small student distilled from the continual teacher
(stage 4, student.pt - it "already learned from the teacher") and adapt it to a NEW domain: real
YouTube clips it never saw.

First we check how the student does on its own UCF101 test set (the in-domain reference), then we
adapt it to the YouTube few-shot task two ways, from the SAME shared init:
- **No meta** - few-shot adapt the student directly.
- **MAML** - first meta-train it over UCF101 N-way K-shot episodes, then few-shot adapt.

Both arms start from the same student and use the same few-shot adaptation, so any gap is due to
episodic meta-training alone.

## **Connect Colab**

In [1]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [2]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

## **Download DATASET**

In [3]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

Mounted at /content/drive
Drive mounted.


100%|██████████| 6.53G/6.53G [05:36<00:00, 20.8MB/s]  

Extracting files...


ULAL_DATASET_ROOT = /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
ULAL_OUTPUT_ROOT  = /content/ucf101-processed/
ULAL_DATA_ROOT     = /content/UCF101
ULAL_DRIVE_PROJECT = /content/drive/MyDrive/apai
Setup complete — no config.py edit, no kernel restart.


In [4]:
!pip install -q yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.2 MB/s eta 0:00:00:00:01


In [5]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
Classes   : base=10 task1=5 task2=5


42

## **Setup**

K-shot is the labelled budget per class at meta-test. N-way is set after download, from the YouTube classes that fetch successfully (a dead URL is skipped).

In [6]:
K_SHOT   = 5
K_QUERY  = 5
EVAL_DRAWS = 10   # random few-shot support draws at meta-test (both arms averaged over these)

META_EPOCHS     = 3      # extra episodic meta-training (MAML arm only)
META_EPISODES   = 40
INNER_LR        = 0.01
INNER_STEPS     = 5
META_LR         = 1e-3
# N_WAY is set after the YouTube download, from the classes that actually downloaded
print(f"{K_SHOT}-shot | {EVAL_DRAWS} eval draws")

4-way 5-shot | 10 eval draws


## **Preprocess data**

In [7]:
base_split  = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
task1_split = build_group_split(cfg.TASK_1,           train_groups=18, val_groups=3)
task2_split = build_group_split(cfg.TASK_2,           train_groups=18, val_groups=3)

In [8]:
# BASE  (max_samples caps TRAIN clips/class via cfg.MAX_SAMPLES; None = full data, val/test untouched)
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT, max_samples=cfg.MAX_SAMPLES)


--- Preprocessing (group-disjoint) ---
To  : /content/UCF101/grouped_base
Classes: 10 | Train limit: Full

Processing split: train



Processing split: val



Processing split: test



Done. Clips ready at: /content/UCF101/grouped_base


In [9]:
# TASK 1  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task1_split, cfg.TASK_1, output_root=cfg.TASK1_ROOT, max_samples=cfg.MAX_SAMPLES)


--- Preprocessing (group-disjoint) ---
To  : /content/UCF101/grouped_task1
Classes: 5 | Train limit: Full

Processing split: train



Processing split: val



Processing split: test



Done. Clips ready at: /content/UCF101/grouped_task1


In [10]:
# TASK 2  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task2_split, cfg.TASK_2, output_root=cfg.TASK2_ROOT, max_samples=cfg.MAX_SAMPLES)


--- Preprocessing (group-disjoint) ---
To  : /content/UCF101/grouped_task2
Classes: 5 | Train limit: Full

Processing split: train



Processing split: val



Processing split: test



Done. Clips ready at: /content/UCF101/grouped_task2


In [11]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx     = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}
print(f"{len(ALL_CLASSES_LIST)} UCF101 classes")

20 UCF101 classes


## **Create Loaders**

A UCF101 test loader (to check the student in-domain) and a RAM pool for the meta-training episodes.

In [12]:
base_train  = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "train"), class_to_idx=class_to_idx)
base_test   = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "test"),  class_to_idx=class_to_idx)
task1_train = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "train"), class_to_idx=class_to_idx)
task1_test  = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "test"),  class_to_idx=class_to_idx)
task2_train = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "train"), class_to_idx=class_to_idx)
task2_test  = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "test"),  class_to_idx=class_to_idx)

test_all_loader = DataLoader(ConcatDataset([base_test, task1_test, task2_test]), batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

In [13]:
pool_clips, pool_labels = build_pool([base_train, task1_train, task2_train], per_class=20)
class_pool = sorted(set(pool_labels.tolist()))
print(f"pool {tuple(pool_clips.shape)}  classes {len(class_pool)}")

pool (400, 16, 3, 224, 224)  classes 20


## **Load the distilled student (learned from the teacher)**

Instead of training a new backbone, load student.pt from stage 4 - the compact MobileNet that
already learned UCF101 by distillation from the continual teacher. This is the shared start for
both arms. (Run the distillation notebook first so student.pt exists in Drive.)

In [ ]:
set_seed(cfg.SEED)
pretrained = MobileNetV3SmallLSTMStudent(num_classes=len(ALL_CLASSES_LIST)).to(device)
pretrained.load_state_dict(torch.load(f"{cfg.CKPT_DIR}/student.pt", map_location=device))
print("loaded distilled student (learned from the teacher)")

## **Test the student on UCF101**

How good is the student on its own 20-class task, before any domain adaptation? This is the
in-domain reference point.

In [ ]:
acc_ucf, preds_ucf, labels_ucf = test_model(pretrained, test_all_loader, device)
print(f"student UCF101 test accuracy: {acc_ucf:.4f}")

In [ ]:
plot_confusion_matrix(labels_ucf, preds_ucf, num_classes=len(ALL_CLASSES_LIST), classes_list=ALL_CLASSES_LIST, name="student on UCF101")

## **YouTube few-shot task**

Download the real out-of-domain YouTube clips. N-way is set from the classes that download (a dead
URL is skipped). Random K-shot support sets are drawn later and both arms are averaged over them.

In [ ]:
yt_clips, yt_labels = download_youtube_clips(cfg.YOUTUBE_CLIPS, class_to_idx, n_clips=40)
yt_ids = sorted(set(yt_labels.tolist()))
N_WAY  = len(yt_ids)     # N-way = the classes that actually downloaded (dead URLs were skipped)
print(f"youtube clips {tuple(yt_clips.shape)}  ->  {N_WAY}-way task")

## **Common init**

Give the student a fresh N-way head for the YouTube classes. Both arms copy this exact model, so
they start identical - the only later difference is the meta-training.

In [ ]:
import copy
init_model = copy.deepcopy(pretrained)
init_model.fc = nn.Linear(cfg.HIDDEN_SIZE, N_WAY).to(device)
print(init_model.fc)

## **Adaptation WITHOUT MAML (no meta)**

Baseline: take the shared init and few-shot adapt it directly to each random YouTube support set
(K_SHOT clips/class, INNER_STEPS SGD steps), then score the query set. Repeated over EVAL_DRAWS
random draws; each draw is seeded so the MAML arm below sees the exact same clips.

In [ ]:
base_accs = []
for d in range(EVAL_DRAWS):
    set_seed(cfg.SEED + d)     # draw d is reproducible and identical in the MAML arm
    sx, sy, qx, qy = make_task(yt_clips, yt_labels, yt_ids, K_SHOT, 1000, device)
    base_accs.append(few_shot_eval(copy.deepcopy(init_model), sx, sy, qx, qy, INNER_LR, INNER_STEPS))
base_mean, base_std = float(np.mean(base_accs)), float(np.std(base_accs))
print(f"no-meta YouTube {N_WAY}-way {K_SHOT}-shot: {base_mean:.4f} +/- {base_std:.4f}")

## **Adaptation WITH MAML**

Same shared init, but first meta-train it over UCF101 N-way K-shot episodes so it learns an init
that adapts fast. Then few-shot adapt to the SAME YouTube draws (same seeds) and score.

In [ ]:
set_seed(cfg.SEED)
maml_model = copy.deepcopy(init_model)
maml_model, hist_maml = meta_train_maml(maml_model, pool_clips, pool_labels, class_pool,
                                        N_WAY, K_SHOT, K_QUERY, META_EPOCHS, META_EPISODES,
                                        INNER_LR, INNER_STEPS, META_LR, device)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, META_EPOCHS + 1), hist_maml["query_accs"], marker="s")
plt.xlabel("meta epoch"); plt.ylabel("meta query accuracy"); plt.ylim(0, 1.05)
plt.title("MAML meta-training progress (UCF101 episodes)")
plt.tight_layout(); plt.show()

In [ ]:
maml_accs = []
for d in range(EVAL_DRAWS):
    set_seed(cfg.SEED + d)     # SAME seed per draw -> SAME support/query as the no-meta arm
    sx, sy, qx, qy = make_task(yt_clips, yt_labels, yt_ids, K_SHOT, 1000, device)
    maml_accs.append(few_shot_eval(copy.deepcopy(maml_model), sx, sy, qx, qy, INNER_LR, INNER_STEPS))
maml_mean, maml_std = float(np.mean(maml_accs)), float(np.std(maml_accs))
print(f"MAML YouTube {N_WAY}-way {K_SHOT}-shot: {maml_mean:.4f} +/- {maml_std:.4f}")
print(f"MAML gain over no-meta: {maml_mean - base_mean:+.4f}  (over {EVAL_DRAWS} draws)")

## **Compare**

The final result: no-meta vs MAML on the YouTube few-shot task, mean +/- std over the draws.
Non-overlapping error bars mean the meta-training genuinely helped.

In [ ]:
methods = ["no meta", "MAML"]
means   = [base_mean, maml_mean]
stds    = [base_std, maml_std]
plt.figure(figsize=(6, 5))
plt.bar(methods, means, yerr=stds, capsize=8)
plt.ylim(0, 1.05); plt.ylabel("YouTube few-shot accuracy")
plt.title(f"{N_WAY}-way {K_SHOT}-shot adaptation to YouTube (mean +/- std over {EVAL_DRAWS} draws)")
plt.tight_layout(); plt.show()

## **Save**

In [ ]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
payload = {"n_way": N_WAY, "k_shot": K_SHOT, "eval_draws": EVAL_DRAWS,
           "shared_init": "distilled_student",
           "student_ucf_test_acc": acc_ucf,
           "no_meta_mean": base_mean, "no_meta_std": base_std,
           "maml_mean": maml_mean, "maml_std": maml_std,
           "no_meta_accs": base_accs, "maml_accs": maml_accs,
           "maml_curve": hist_maml["query_accs"]}
with open(f"{cfg.RESULTS_DIR}/stage5_meta_domain_adapt.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved -> {cfg.RESULTS_DIR}/stage5_meta_domain_adapt.json")